In [1]:
!pip install lm-eval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.1 MB/s  0:00:00
  Created wheel for sqlitedict: filename=sqlitedict-2.1.0-py3-none-any.whl size=16957 sha256=e8766e3273ccb7f6040060764da9418ef546d5d9c2faf2148298530b6b54b660
  Stored in directory: /root/.cache/pip/wheels/56/1e/68/fc282426da812281e81368e59ed19f3edc6d9eef4518dbc41c
  Created wheel for word2number: filename=word2number-1.1-py3-none-any.whl size=5659 sha256=90de18a2ee4cdc265576816e08035a6d8026990b4ae1d96f928742fc7c5ea040
  Stored in directory: /root/.cache/pip/wheels/aa/2b/71/ede4a3c3520a8374778c16c345e0e78e25d1ffe38f4c5c1e8a
Successfully built sqlitedict word2number
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [lm-eval]2/13 [lm-eval]riter]


In [2]:
import lm_eval
import torch
import math
import json
import time
import glob

from tokenizers import Tokenizer
from lm_eval.api.model import LM
from tqdm import tqdm
print(lm_eval.__version__)

0.4.13


In [3]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_RL = '/kaggle/input/datasets/absolute1000/model-after-rl/RL_BEST.pt'
MODEL_SFT = '/kaggle/input/datasets/absolute1000/model-after-sft/SFT_BEST.pt'
TOKENIZER = '/kaggle/input/datasets/absolute1000/tokenized-llm-training/tokenizer.json'

In [4]:
def RoPE(batch):
    B, L, D = batch.shape
    x1 = batch[..., 0::2]
    x2 = batch[..., 1::2]

    pos = torch.arange(L, device=batch.device)

    freqs = 10000 ** (-torch.arange(0, D, 2, device=batch.device) / D)

    angles = pos[:, None] * freqs[None, :]

    cos = torch.cos(angles).unsqueeze(0)
    sin = torch.sin(angles).unsqueeze(0)

    x1_rot = x1 * cos - x2 * sin
    x2_rot = x1 * sin + x2 * cos

    return torch.stack([x1_rot, x2_rot], dim=-1).flatten(-2)

class SelfAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, x, mask=None): # x[batch_size, tok_size, emb_size], 
        query, key, value = self.Query(x), self.Key(x), self.Value(x) #[batch, tok_size, head_size]
        query, key = RoPE(query), RoPE(key)
        
        internal_state = query @ key.transpose(-2, -1) #[batch, tok_size, tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1) #[batch, tok_size, tok_size]
        out = weights @ value
        return out # [batch, tok_size, head_size]

class MultiHeadSelfAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([SelfAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, x, mask=None):
        temporary_states = [head(x, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, tok_size, head_size] -> [batch, tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, tok_size, emb_size]

class FeedForward(torch.nn.Module):
    def __init__(self, emb_size, out_size=None):
        super().__init__()
        neurons = int((8 / 3) * emb_size) - (int((8 / 3) * emb_size) % 64)
        if out_size is None: out_size = emb_size
        self.W1 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.SiLU = torch.nn.SiLU()
        self.W2 = torch.nn.Linear(emb_size, neurons, bias=False)
        self.W3 = torch.nn.Linear(neurons, out_size, bias=False)

    def forward(self, x): # [batch, tok_size, emb_size]
        hidden_state = self.SiLU(self.W1(x)) * self.W2(x) # [batch, tok_size, neurons]
        output = self.W3(hidden_state)
        return output # [batch, tok_size, out_size]

class EncoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask): # [batch_size, context_tok_size, emb_size]

        # [batch_size, context_tok_size, emb_size]
        residual_connection1 = self.MultiHeadSelfAttention(self.RMSNorm1(context), mask) + context
        
        # [batch_size, context_tok_size, emb_size]
        residual_connection2 = self.FeedForward(self.RMSNorm2(residual_connection1)) + residual_connection1

        return residual_connection2 # [batch_size, context_tok_size, emb_size]

class Encoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([EncoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)

    def forward(self, context, mask):
        internal_state = context
        for encoder in self.blocks:
            internal_state = encoder(internal_state, mask)
        output = self.RMSNorm(internal_state)
        return output # [batch_size, context_tok_size, emb_size]

class Thinking(torch.nn.Module):
    def __init__(self, emb_size, internal_state_size=128, vocab_size=12288, momentum=0.1):
        super().__init__()
        self.momentum = momentum
        self.to_vocab = torch.nn.Linear(emb_size, vocab_size, bias=False)
        self.alpha = torch.nn.Parameter(torch.ones(vocab_size))
        self.SiLU = torch.nn.SiLU()

        self.to_internal = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.ln = torch.nn.LayerNorm(internal_state_size)
        self.to_emb = torch.nn.Linear(internal_state_size, emb_size, bias=False)

        self.register_buffer("e_mean", torch.zeros(vocab_size))
        self.t_in = torch.nn.Linear(vocab_size, internal_state_size, bias=False)
        self.t_ln = torch.nn.LayerNorm(internal_state_size)
        self.t_out = torch.nn.Linear(internal_state_size, emb_size, bias=False)

    def forward(self, x, mask_for_thinking, context_ids):
        # x [B, Cs, Emb], mask_for_thinking [B, Cs, 1], context_ids [B, Cs]
        x_unit = x / x.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        vocab_unit = self.to_vocab.weight / self.to_vocab.weight.norm(dim=-1, keepdim=True).clamp(min=1e-6)
        energy = x_unit @ vocab_unit.T # [B, Cs, V], from -1 to 1

        own = torch.zeros_like(energy, dtype=torch.bool).scatter_(2, context_ids[..., None], True)
        drop = (~mask_for_thinking) | own # padding + the position's own token
        kept = energy.masked_fill(drop, 0.0) # [B, Cs, V]

        # words channel
        internal_state = kept * self.alpha
        internal_state = self.to_internal(internal_state)
        internal_state = self.ln(internal_state)
        internal_state = self.SiLU(internal_state)
        words = self.to_emb(internal_state) # [B, Cs, Emb]

        # thought channel
        e = kept.sum(dim=1).float() # [B, V]: sum of energies over positions
        if self.training:
            mu = e.mean(dim=0) # mean over texts in the batch
            with torch.no_grad():
                self.e_mean.mul_(1 - self.momentum).add_(self.momentum * mu)
        else:
            mu = self.e_mean # inference: running mean
        e = e - mu # how this text differs from the others
        e = e / e.pow(2).mean(dim=-1, keepdim=True).sqrt().clamp(min=1e-6) # unit scale

        thought = self.t_in(e)
        thought = self.t_ln(thought)
        thought = self.SiLU(thought)
        thought = self.t_out(thought) # [B, Emb]

        thinking_out = words + thought.unsqueeze(1) # thought is added to every position
        return thinking_out

class CrossAttention(torch.nn.Module):
    def __init__(self, emb_size, head_size):
        super().__init__()
        self.C = math.sqrt(head_size)
        self.Query = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Key = torch.nn.Linear(emb_size, head_size, bias=False)
        self.Value = torch.nn.Linear(emb_size, head_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        # data_for_query[batch, answer_tok_size, emb_size]
        # data_for_key_value[batch, context_tok_size, emb_size] 
        
        query, key, value = self.Query(data_for_query), self.Key(data_for_key_value), self.Value(data_for_key_value)
        # query[batch, answer_tok_size, head_size]
        # key, value[batch, context_tok_size, head_size]
        
        internal_state = query @ key.transpose(-2, -1) #[batch, answer_tok_size, context_tok_size]
        if mask is not None: internal_state = internal_state.masked_fill(~mask, float('-inf'))
        weights = torch.nn.functional.softmax(internal_state / self.C, dim=-1)
        out = weights @ value
        return out #[batch, answer_tok_size, head_size]

class MultiHeadCrossAttention(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        head_size = emb_size // n_heads
        self.heads = torch.nn.ModuleList([CrossAttention(emb_size, head_size) for head in range(n_heads)])
        self.Output = torch.nn.Linear(emb_size, emb_size, bias=False)

    def forward(self, data_for_query, data_for_key_value, mask=None):
        temporary_states = [head(data_for_query, data_for_key_value, mask) for head in self.heads]
        temporary_states = torch.cat(temporary_states, dim=-1) # [batch, answer_tok_size, head_size] -> [batch, answer_tok_size, emb_size]
        output = self.Output(temporary_states)
        return output # [batch, answer_tok_size, emb_size]

class DecoderBlock(torch.nn.Module):
    def __init__(self, emb_size, n_heads):
        super().__init__()
        self.MultiHeadSelfAttention = MultiHeadSelfAttention(emb_size, n_heads)
        self.RMSNorm1 = torch.nn.RMSNorm(emb_size)
        self.MultiHeadCrossAttention = MultiHeadCrossAttention(emb_size, n_heads)
        self.RMSNorm2 = torch.nn.RMSNorm(emb_size)
        self.RMSNorm3 = torch.nn.RMSNorm(emb_size)
        self.FeedForward = FeedForward(emb_size)

    def forward(self, encoder_output, answer, encoder_mask=None):
        
        # [batch, answer_tok_size, emb_size]
        internal_state = self.MultiHeadSelfAttention(self.RMSNorm1(answer))
        residual_connection1 =  internal_state + answer
        internal_state = self.MultiHeadCrossAttention(
            self.RMSNorm2(residual_connection1), encoder_output, encoder_mask) # [batch, answer_tok_size, emb_size]
        residual_connection2 = internal_state + residual_connection1
        internal_state = self.FeedForward(self.RMSNorm3(residual_connection2))
        residual_connection3 = internal_state + residual_connection2 # [batch, answer_tok_size, emb_size]
        return residual_connection3 # [batch, answer_tok_size, emb_size]
        
class Decoder(torch.nn.Module):
    def __init__(self, n_blocks, emb_size, n_heads):
        super().__init__()
        self.blocks = torch.nn.ModuleList([DecoderBlock(emb_size, n_heads) for block in range(n_blocks)])
        self.RMSNorm = torch.nn.RMSNorm(emb_size)
        self.final = torch.nn.Linear(emb_size, 12288, bias=False)

    def forward(self, answer, encoder_output, mask):
        internal_state = answer
        for decoder in self.blocks:
            internal_state = decoder(encoder_output, internal_state, mask) # [batch_size, answer_size, emb_size]
        internal_state = self.RMSNorm(internal_state)
        internal_state = self.final(internal_state)
        return internal_state # [batch_size, answer_size, dict_size]

class NN(torch.nn.Module):
    def _init_weights(self, m):
        if isinstance(m, (torch.nn.Linear, torch.nn.Embedding)):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
        
    def __init__(self, 
                 embedding_size, 
                 encoder_num, 
                 heads_encoder,
                 thinking_internal_state_size, 
                 decoder_num, 
                 heads_decoder):
        super().__init__()
        
        self.Embedding = torch.nn.Embedding(12288, embedding_size)
        self.Encoder = Encoder(encoder_num, embedding_size, heads_encoder)
        self.Thinking = Thinking(embedding_size, thinking_internal_state_size)
        self.Thinking.to_vocab.weight = self.Embedding.weight
        self.out_norm = torch.nn.RMSNorm(embedding_size)
        self.Decoder = Decoder(decoder_num, embedding_size, heads_decoder)
        self.Decoder.final.weight = self.Embedding.weight
        self.apply(self._init_weights)
        torch.nn.init.zeros_(self.Thinking.t_out.weight)

    def encode(self, context_ids):
        # context_ids [B, Cs]
        PAD_ID = 0
        mask = (context_ids != PAD_ID)[:, None, :] # [B, 1, Cs]
        memory = self.Encoder(self.Embedding(context_ids), mask) # [B, Cs, Emb]
        memory = self.Thinking(memory, mask.transpose(1, 2), context_ids)
        memory = self.out_norm(memory)
        return memory, mask
    
    def decode(self, answer_ids, memory, mask):
        return self.Decoder(self.Embedding(answer_ids), memory, mask) # [B, La, V]
    
    def forward(self, context_ids, answer_ids):
        memory, mask = self.encode(context_ids)
        return self.decode(answer_ids, memory, mask)

In [5]:
def collate_fn(list_of_samples):
    #tok.encode("<pad>").ids = [0]
    
    B = len(list_of_samples)
    _batch_samples = torch.zeros((B, 256), dtype=torch.int64)
    _batch_targets = torch.zeros((B, 64), dtype=torch.int64)
    for idx, data in enumerate(list_of_samples):
        context, bot_tokens = data
        _batch_samples[idx, -len(context):] = torch.tensor(context)
        _batch_targets[idx, :len(bot_tokens)] = torch.tensor(bot_tokens)
    return _batch_samples, _batch_targets

In [6]:
def encode_pair(context, continuation):
    ctx_cut, ans_cut = False, False
    l = len(context) - len(context.rstrip())
    if l > 0:
        continuation = context[-l:] + continuation
        context = context[:-l]

    ctx_ids = tok.encode(context).ids if len(context) != 0 else [tok.encode('<eos>').ids[0]]
    ans_ids = tok.encode(continuation).ids

    if len(ctx_ids) > 256: 
        ctx_ids = ctx_ids[-256:]
        ctx_cut = True
    if len(ans_ids) > 64:
        ans_ids = ans_ids[:64]
        ans_cut = True
    ctx_ids = [tok.encode('<pad>').ids[0]] * (256 - len(ctx_ids)) + ctx_ids

    return ctx_ids, ans_ids, ctx_cut, ans_cut    

In [7]:
@torch.no_grad()
def score_chunk(model, chunk, cur_rows_per_sample, ans_size=64):
    _ = torch.tensor(cur_rows_per_sample, device=DEVICE)
    batch_ctx, batch_tgt = collate_fn(chunk) #[B, 256], [B, 64]
    batch_ctx, batch_tgt = batch_ctx.to(DEVICE), batch_tgt.to(DEVICE)
    
    KV_cache, mask = model.encode(batch_ctx) # [B, Cs, Emb], [B, 1, Cs]
    KV_cache = KV_cache.repeat_interleave(_, dim=0) # [cur_rows, Cs, Emb]
    mask = mask.repeat_interleave(_, dim=0) #[cur_rows, 1, Cs]
    slots = batch_tgt.repeat_interleave(_, dim=0) #[cur_rows, 64]

    start_masking = []
    for num_rows in _:
        masking_from_this_token = [i for i in range(num_rows)]
        start_masking += masking_from_this_token
    tgt_tok = slots[torch.arange(len(start_masking), device=DEVICE), start_masking]

    mask_ = torch.tensor(start_masking, device=DEVICE).unsqueeze(1) <= torch.arange(ans_size, device=DEVICE)[None, :]
    _ = torch.arange(len(start_masking), device=DEVICE)
    
    slots = slots.masked_fill(mask_, tok.encode('<mask>').ids[0])
    logits = model.decode(slots, KV_cache, mask) #[cur_rows, 64, 12288]

    logprobs = logits[_, start_masking].float().log_softmax(-1) #[cur_rows, 12288]
    lp = logprobs[_, tgt_tok]
    
    greedy = logprobs.argmax(-1) == tgt_tok
    gr_pair = torch.split(greedy, cur_rows_per_sample)
    lp_pair = torch.split(lp, cur_rows_per_sample)

    ans = []
    for idx in range(len(cur_rows_per_sample)):
        ans.append((lp_pair[idx].sum().item(), gr_pair[idx].all().item()))
    return ans

def score_pairs(model, pairs, slot_len=64, max_rows=512, max_pairs=64):
    model.eval()
    cur_rows = []
    start = 0
    results = []
    for i in tqdm(range(len(pairs)), mininterval=30):
        if sum(cur_rows) + len(pairs[i][1]) <= max_rows and len(cur_rows) < max_pairs:
            cur_rows.append(len(pairs[i][1]))
        else:
            results.extend(score_chunk(model, pairs[start:i], cur_rows, slot_len))
            start = i
            cur_rows = [len(pairs[i][1])]
    if cur_rows:
        results.extend(score_chunk(model, pairs[start:], cur_rows, slot_len))
    return results

In [8]:
class DiffusionLM(LM):
    def __init__(self, model, slot_len=64, max_rows=512, max_pairs=64):
        super().__init__()
        self.m = model
        self.ctx_cut = 0
        self.ans_cut = 0
        self.rows = 0
        self.slot_len = slot_len
        self.max_rows = max_rows
        self.max_pairs = max_pairs

    def generate_until(self, requests):
        raise NotImplementedError

    def loglikelihood_rolling(self, requests):
        raise NotImplementedError

    def loglikelihood(self, requests):
        self.ctx_cut, self.ans_cut, self.rows = 0, 0, 0
        self.pairs = len(requests)
        buffer = []
        for r in requests:
            ctx_ids, ans_ids, ctx_cut, ans_cut = encode_pair(*r.args)
            self.ctx_cut += ctx_cut
            self.ans_cut += ans_cut
            self.rows += len(ans_ids)
            buffer.append((ctx_ids, ans_ids))
        res = score_pairs(self.m, buffer, self.slot_len, self.max_rows, self.max_pairs)
        return res

In [9]:
MODELS = {'sft': MODEL_SFT, 'rl': MODEL_RL}
for m in MODELS:
    ckpt = torch.load(MODELS[m], map_location=DEVICE, weights_only=False)
    MODELS[m] = NN(**ckpt['MODEL_SETTINGS_CONFIG']).to(DEVICE)
    MODELS[m].load_state_dict(ckpt['model'], strict=True)
    MODELS[m].eval()

tok = Tokenizer.from_file(TOKENIZER)

In [10]:
tasks = ['piqa', 'hellaswag', 'arc_easy', 'arc_challenge', 'winogrande', 'sciq', 'lambada_openai', 'blimp']

In [11]:
def make_record(model_type, ctx_cut, ans_cut, pairs, rows, benchmark_name, res, version, slot_len, max_rows,
               n_params, time):
    path = f'/kaggle/working/{'_'.join([model_type.upper(), benchmark_name])}.json'
    #Path(path).unlink(missing_ok=True)
    rec = {
        'model_type':model_type,
        'benchmark_name':benchmark_name,
        'res':res,
        'ctx_cut':ctx_cut,
        'ans_cut':ans_cut,
        'pairs':pairs,
        'rows':rows,
        'lm_eval_version':version,
        'slot_len':slot_len,
        'max_rows':max_rows,
        'n_params':n_params,
        'time':time
    }
    with open(path, 'w', encoding='UTF-8') as f:
        json.dump(rec, f, default=str)

In [12]:
def get_metric(results, task):
    d = 0
    if task in ['hellaswag', 'arc_easy', 'arc_challenge']:
        d = results[task]['acc_norm,none']
    elif task == 'blimp' and results['blimp'].get('acc,none', None) is None:
        c = 0
        for k in results:
            if k.startswith('blimp_'):
                c += 1
                d += results[k]['acc,none']
        d /= c
    elif task in ['piqa', 'winogrande', 'sciq', 'blimp', 'lambada_openai']:
        d = results[task]['acc,none']
    return d

In [13]:
for mt in MODELS:
    model = DiffusionLM(MODELS[mt], 64, 2048, 64)
    for task in tasks:
        t0 = time.time()
        res = lm_eval.simple_evaluate(model=model, tasks=[task], num_fewshot=0, log_samples=False)
        t = time.time() - t0
        make_record(mt, model.ctx_cut, model.ans_cut, model.pairs, model.rows, task, res["results"], lm_eval.__version__,
                   model.slot_len, model.max_rows, sum(p.numel() for p in model.m.parameters()), t)
        print(f"{mt} {task}: {get_metric(res['results'], task) * 100:.1f}  "
              f"обрезано {model.ans_cut / model.pairs * 100:.1f}%  {t:.0f} c", flush=True)
    del model
    torch.cuda.empty_cache()

piqa_train.parquet: reconstructing file:   0%|          |  0.00B / 2.64MB            

piqa_train.parquet: downloading bytes:           |  0.00B            

piqa_validation.parquet: reconstructing file:   0%|          |  0.00B /  300kB            

piqa_validation.parquet: downloading bytes:           |  0.00B            

piqa_test.parquet: reconstructing file:   0%|          |  0.00B /  496kB            

...

bootstrapping for stddev: perplexity


100%|██████████| 100/100 [00:23<00:00,  4.21it/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


rl lambada_openai: 0.0  обрезано 0.0%  74 c


100%|██████████| 134000/134000 [22:13<00:00, 100.47it/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


rl blimp: 58.5  обрезано 0.0%  1560 c


In [14]:
!lm_eval --model hf --model_args pretrained=EleutherAI/pythia-14m,dtype=float32 --tasks piqa,hellaswag,arc_easy,arc_challenge,winogrande,sciq,lambada_openai,blimp --num_fewshot 0 --batch_size auto --device cuda --output_path /kaggle/working/baselines/
!lm_eval --model hf --model_args pretrained=EleutherAI/pythia-70m,dtype=float32 --tasks piqa,hellaswag,arc_easy,arc_challenge,winogrande,sciq,lambada_openai,blimp --num_fewshot 0 --batch_size auto --device cuda --output_path /kaggle/working/baselines/
!lm_eval --model hf --model_args pretrained=openai-community/gpt2,dtype=float32 --tasks piqa,hellaswag,arc_easy,arc_challenge,winogrande,sciq,lambada_openai,blimp --num_fewshot 0 --batch_size auto --device cuda --output_path /kaggle/working/baselines/
!lm_eval --model hf --model_args pretrained=HuggingFaceTB/SmolLM2-135M,dtype=float32 --tasks piqa,hellaswag,arc_easy,arc_challenge,winogrande,sciq,lambada_openai,blimp --num_fewshot 0 --batch_size auto --device cuda --output_path /kaggle/working/baselines/
!lm_eval --model hf --model_args pretrained=HuggingFaceTB/SmolLM2-135M-Instruct,dtype=float32 --tasks piqa,hellaswag,arc_easy,arc_challenge,winogrande,sciq,lambada_openai,blimp --num_fewshot 0 --batch_size auto --device cuda --output_path /kaggle/working/baselines/

2026-10-23:18:55:54 INFO     [_cli.run:392] Selected Tasks: ['piqa', 'hellaswag', 'arc_easy', 'arc_challenge', 'winogrande', 'sciq', 'lambada_openai', 'blimp']
2026-10-23:18:55:56 INFO     [evaluator:214] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-10-23:18:55:56 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': 'EleutherAI/pythia-14m', 'dtype': 'float32'}
2026-10-23:18:56:06 INFO     [models.huggingface:289] Using device 'cuda'
config.json: 100%|█████████████████████████████| 698/698 [00:00<00:00, 3.04MB/s]
tokenizer_config.json: 100%|███████████████| 4.83k/4.83k [00:00<00:00, 10.3MB/s]
tokenizer.json: 100%|██████████████████████| 2.11M/2.11M [00:00<00:00, 45.2MB/s]
special_tokens_map.json: 100%|█████████████████| 441/441 [00:00<00:00, 2.20MB/s]
2026-10-23:18:56:07 INFO     [models.huggingface:582] Model parallel was set to False, max memory was not set, and device map

In [15]:
public_models = ['EleutherAI__pythia-14m', 'EleutherAI__pythia-70m', 'openai-community__gpt2', 
          'HuggingFaceTB__SmolLM2-135M', 'HuggingFaceTB__SmolLM2-135M-Instruct']
pm_params = {'EleutherAI__pythia-14m': '14M', 'EleutherAI__pythia-70m': '70M', 'openai-community__gpt2': '124M',
'HuggingFaceTB__SmolLM2-135M': '135M', 'HuggingFaceTB__SmolLM2-135M-Instruct': '135M'}

In [16]:
final_table = dict()

for task in tasks:
    for m in ['SFT', 'RL']:
        path = '/kaggle/working/' + m + '_' + task + '.json'

        with open(path, 'r', encoding='UTF-8') as f:
            rec = json.load(f)
            res = get_metric(rec['res'], task)
            cut = rec['ans_cut'] / rec['pairs'] * 100
            params_num = rec['n_params']
            final_table.setdefault(m, {})[task] = {'res': res, 'cut': cut, 'params_num': params_num}

for m in public_models:  
    path = '/kaggle/working/baselines/' + m + '/results_*.json'
    with open(sorted(glob.glob(path))[-1], 'r', encoding='UTF-8') as f:
        rec = json.load(f)
        for task in tasks:
            res = get_metric(rec['results'], task)
            params_num = pm_params[m]
            final_table.setdefault(m, {})[task] = {'res':res, 'params_num':params_num}        

In [17]:
from IPython.display import Markdown, display

TASK_NAMES = {'piqa': 'PIQA', 'hellaswag': 'HellaSwag', 'arc_easy': 'ARC-e', 'arc_challenge': 'ARC-c',
              'winogrande': 'WinoG', 'sciq': 'SciQ', 'lambada_openai': 'LAMBADA', 'blimp': 'BLiMP'}
NORM_TASKS = {'hellaswag', 'arc_easy', 'arc_challenge'}
CHANCE = {'piqa': 50.0, 'hellaswag': 25.0, 'arc_easy': 25.0, 'arc_challenge': 25.0,
          'winogrande': 50.0, 'sciq': 25.0, 'lambada_openai': 0.0, 'blimp': 50.0}
OURS = {'SFT': 'ours · SFT', 'RL': 'ours · RL'}


def fmt_params(n):
    if isinstance(n, str):
        return n
    return f"{n / 1e6:.1f}M" if n < 1e9 else f"{n / 1e9:.2f}B"


def md_table(header, rows, align):
    widths = [max(len(str(r[i])) for r in [header] + rows) for i in range(len(header))]
    def line(cells):
        return '| ' + ' | '.join(str(c).ljust(w) if a == 'l' else str(c).rjust(w)
                                 for c, w, a in zip(cells, widths, align)) + ' |'
    sep = '|' + '|'.join((':' + '-' * (w + 1)) if a == 'l' else ('-' * (w + 1) + ':')
                         for w, a in zip(widths, align)) + '|'
    return '\n'.join([line(header), sep] + [line(r) for r in rows])


models = [m for m in OURS if m in final_table] + [m for m in final_table if m not in OURS]
names = {m: OURS.get(m, m.replace('__', '/')) for m in models}
best = {t: max(final_table[m][t]['res'] for m in models) for t in tasks}

# results
header = ['Model', 'Params'] + [TASK_NAMES[t] + ('*' if t in NORM_TASKS else '') for t in tasks]
rows = []
for m in models:
    cells = [names[m], fmt_params(final_table[m][tasks[0]]['params_num'])]
    for t in tasks:
        v = final_table[m][t]['res']
        s = f"{v * 100:.1f}"
        cells.append(f"**{s}**" if v == best[t] else s)
    rows.append(cells)
rows.append(['*random*', '—'] + [f"{CHANCE[t]:.1f}" for t in tasks])
results_md = md_table(header, rows, ['l', 'r'] + ['r'] * len(tasks))

# cuts
header2 = ['Model'] + [TASK_NAMES[t] for t in tasks]
rows2 = [[names[m]] + [f"{final_table[m][t]['cut']:.1f}" for t in tasks] for m in OURS if m in final_table]
cut_md = md_table(header2, rows2, ['l'] + ['r'] * len(tasks))

report = '\n'.join([
    '## Benchmark results',
    '',
    results_md,
    '',
    f"Zero-shot, lm-evaluation-harness {lm_eval.__version__}. Accuracy, %; "
    "\\* — length-normalized accuracy (`acc_norm`). **Bold** — best in column. "
    "Ours: context 256, answer slot 64, diffusion likelihood via left-to-right unmasking.",
    '',
    '### Truncated answer options, % (slot = 64 tokens)',
    '',
    cut_md,
    '',
])

with open('/kaggle/working/results.md', 'w', encoding='UTF-8') as f:
    f.write(report)
print(report)
display(Markdown(report))

## Benchmark results

| Model                               | Params |     PIQA | HellaSwag* |   ARC-e* |   ARC-c* |    WinoG |     SciQ |  LAMBADA |    BLiMP |
|:------------------------------------|-------:|---------:|-----------:|---------:|---------:|---------:|---------:|---------:|---------:|
| ours · SFT                          |  19.8M |     52.6 |       26.0 |     30.3 |     22.4 |     51.9 |     33.7 |      0.0 |     58.9 |
| ours · RL                           |  19.8M |     52.8 |       25.8 |     30.7 |     22.2 |     50.0 |     29.5 |      0.0 |     58.5 |
| EleutherAI/pythia-14m               |    14M |     56.9 |       26.5 |     32.7 |     21.0 |     51.3 |     50.4 |     10.3 |     67.7 |
| EleutherAI/pythia-70m               |    70M |     59.8 |       27.4 |     35.2 |     21.8 |     52.9 |     64.0 |     22.7 |     74.2 |
| openai-community/gpt2               |   124M |     62.9 |       31.1 |     39.5 |     22.7 |     51.6 |     75.2 |     32.6 | **82.1** |
| Hug

## Benchmark results

| Model                               | Params |     PIQA | HellaSwag* |   ARC-e* |   ARC-c* |    WinoG |     SciQ |  LAMBADA |    BLiMP |
|:------------------------------------|-------:|---------:|-----------:|---------:|---------:|---------:|---------:|---------:|---------:|
| ours · SFT                          |  19.8M |     52.6 |       26.0 |     30.3 |     22.4 |     51.9 |     33.7 |      0.0 |     58.9 |
| ours · RL                           |  19.8M |     52.8 |       25.8 |     30.7 |     22.2 |     50.0 |     29.5 |      0.0 |     58.5 |
| EleutherAI/pythia-14m               |    14M |     56.9 |       26.5 |     32.7 |     21.0 |     51.3 |     50.4 |     10.3 |     67.7 |
| EleutherAI/pythia-70m               |    70M |     59.8 |       27.4 |     35.2 |     21.8 |     52.9 |     64.0 |     22.7 |     74.2 |
| openai-community/gpt2               |   124M |     62.9 |       31.1 |     39.5 |     22.7 |     51.6 |     75.2 |     32.6 | **82.1** |
| HuggingFaceTB/SmolLM2-135M          |   135M | **68.4** |   **43.1** | **58.6** | **29.6** | **53.0** | **84.0** | **42.9** |     79.0 |
| HuggingFaceTB/SmolLM2-135M-Instruct |   135M |     66.7 |       42.9 |     48.8 |     27.7 |     51.1 |     81.8 |     39.7 |     74.6 |
| *random*                            |      — |     50.0 |       25.0 |     25.0 |     25.0 |     50.0 |     25.0 |      0.0 |     50.0 |

Zero-shot, lm-evaluation-harness 0.4.13. Accuracy, %; \* — length-normalized accuracy (`acc_norm`). **Bold** — best in column. Ours: context 256, answer slot 64, diffusion likelihood via left-to-right unmasking.

### Truncated answer options, % (slot = 64 tokens)

| Model      | PIQA | HellaSwag | ARC-e | ARC-c | WinoG | SciQ | LAMBADA | BLiMP |
|:-----------|-----:|----------:|------:|------:|------:|-----:|--------:|------:|
| ours · SFT |  8.5 |       1.7 |   0.0 |   0.0 |   0.0 |  0.0 |     0.0 |   0.0 |
| ours · RL  |  8.5 |       1.7 |   0.0 |   0.0 |   0.0 |  0.0 |     0.0 |   0.0 |
